# 📊 01 — Análise Exploratória de Dados (EDA)

> **Credit Risk ML** · Notebook 1 de 5

**Objetivo:** entender a estrutura dos dados antes de qualquer modelagem.  
Boas decisões de engenharia de features e escolha de modelos nascem aqui.

| Etapa | Descrição |
|-------|-----------|
| 1.1 | Carregamento e visão geral |
| 1.2 | Qualidade dos dados (nulos, tipos, duplicatas) |
| 1.3 | Distribuição do target |
| 1.4 | Distribuições univariadas |
| 1.5 | Análise bivariada (feature × default) |
| 1.6 | Correlações |
| 1.7 | Insights e decisões |

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# ── Estilo ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0F1117', 'axes.facecolor': '#161B22',
    'axes.edgecolor':   '#30363D', 'axes.labelcolor': '#E6EDF3',
    'xtick.color':      '#8B949E', 'ytick.color':     '#8B949E',
    'text.color':       '#E6EDF3', 'grid.color':      '#21262D',
    'grid.linewidth':   0.8,       'figure.dpi':      120,
    'font.family':      'monospace',
})
C = {'good': '#3FB950', 'bad': '#F78166', 'neutral': '#58A6FF',
     'accent': '#D2A8FF', 'warn': '#E3B341', 'muted': '#8B949E'}

SEED = 42
np.random.seed(SEED)
print('✅  Imports OK')

---
### 1.1 Carregamento dos Dados

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Para datasets reais do Kaggle:
#   df = pd.read_csv('/kaggle/input/seu-dataset/credit.csv')
# ─────────────────────────────────────────────────────────────────

DATA_PATH = Path('../data/synthetic/credit_data.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    # Gera inline se o arquivo não existir
    rng = np.random.default_rng(SEED)
    n   = 50_000
    income       = rng.lognormal(8.5, 0.6, n).round(2)
    loan_amount  = (income * rng.uniform(0.5, 5.0, n)).round(2)
    loan_tenure  = rng.choice([12,24,36,48,60,72], n)
    credit_score = np.clip(rng.normal(650,100,n),300,850).round().astype(int)
    emp_years    = np.clip(rng.exponential(5,n),0,40).round(1)
    dti          = np.clip((loan_amount/loan_tenure)/(income/12),0.01,2.0).round(4)
    log_odds     = -2.5 + (-0.006)*credit_score + 1.5*dti + 0.12*rng.integers(0,10,n) \
                   - 0.04*emp_years - 0.003*(income/1000) + 0.3*(loan_amount/loan_amount.mean())
    default      = rng.binomial(1, 1/(1+np.exp(-log_odds))).astype(int)
    df = pd.DataFrame({
        'age': rng.integers(21,70,n), 'income': income,
        'loan_amount': loan_amount, 'loan_tenure': loan_tenure,
        'credit_score': credit_score, 'num_open_accounts': rng.integers(1,15,n),
        'num_credit_inq': rng.integers(0,10,n), 'debt_to_income': dti,
        'employment_years': emp_years,
        'has_mortgage': rng.integers(0,2,n), 'has_dependents': rng.integers(0,2,n),
        'default': default,
    })
    for col in ['income','credit_score','employment_years']:
        df.loc[rng.random(n)<0.03, col] = np.nan

print(f'✅  Dataset carregado')
print(f'   Shape       : {df.shape}')
print(f'   Memória     : {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
print(f'   Default rate: {df["default"].mean():.1%}')
df.head()

---
### 1.2 Qualidade dos Dados

In [ ]:
print('── TIPOS E NULOS ───────────────────────────────────')
summary = pd.DataFrame({
    'dtype'  : df.dtypes,
    'nulos'  : df.isnull().sum(),
    'nulos%' : (df.isnull().mean()*100).round(2),
    'únicos' : df.nunique(),
    'min'    : df.min(),
    'max'    : df.max(),
})
display(summary)

n_dup = df.duplicated().sum()
print(f'\n  Duplicatas : {n_dup}')
print(f'  Linhas sem target: {df["default"].isna().sum()}')

In [ ]:
# Mapa visual de valores nulos
nulls = df.isnull()
if nulls.any().any():
    fig, ax = plt.subplots(figsize=(12, 3))
    fig.patch.set_facecolor('#0F1117')
    sns.heatmap(nulls.T, cmap=['#161B22', C['bad']], cbar=False,
                yticklabels=True, xticklabels=False, ax=ax)
    ax.set_title('Mapa de Valores Nulos (amarelo = nulo)', color='#E6EDF3', fontsize=12)
    plt.tight_layout(); plt.show()
else:
    print('✅  Nenhum valor nulo encontrado')

---
### 1.3 Distribuição do Target

In [ ]:
counts  = df['default'].value_counts()
default_rate = df['default'].mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#0F1117')

# Barras
ax = axes[0]
bars = ax.bar(['Bom (0)','Default (1)'], counts.values,
              color=[C['good'], C['bad']], edgecolor='#0F1117', width=0.5)
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
            f'{v:,}', ha='center', fontsize=12, fontweight='bold')
ax.set_title('Contagem por Classe', fontsize=12, color='#E6EDF3')
ax.grid(axis='y', alpha=0.3)

# Proporção
ax2 = axes[1]
ax2.pie([counts[0],counts[1]], labels=['Bom','Default'],
        colors=[C['good'],C['bad']], autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor':'#0F1117','linewidth':3},
        pctdistance=0.7)
ax2.set_title('Proporção de Classes', fontsize=12, color='#E6EDF3')

# Taxa de default acumulada (simulando chegada de dados ao longo do tempo)
ax3 = axes[2]
cumulative_rate = df['default'].expanding().mean()
ax3.plot(cumulative_rate.values[::100], lw=1.5, color=C['bad'])
ax3.axhline(default_rate, lw=1.5, ls='--', color=C['warn'],
            label=f'Média = {default_rate:.1%}')
ax3.set_title('Taxa de Default Acumulada', fontsize=12, color='#E6EDF3')
ax3.set_xlabel('Amostras (×100)'); ax3.set_ylabel('Taxa')
ax3.legend(fontsize=9, framealpha=0.3); ax3.grid(alpha=0.2)

plt.suptitle('Análise do Target — Desbalanceamento de Classes',
             fontsize=14, color=C['accent'], fontweight='bold')
plt.tight_layout(); plt.show()

print(f'  Ratio classes (bom:default) = {counts[0]/counts[1]:.1f}:1')
print(f'  → Usar class_weight="balanced" ou scale_pos_weight={counts[0]/counts[1]:.1f}')

---
### 1.4 Distribuições Univariadas

In [ ]:
num_cols = ['age','income','loan_amount','loan_tenure','credit_score',
            'num_open_accounts','num_credit_inq','debt_to_income','employment_years']

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.patch.set_facecolor('#0F1117')
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    data = df[col].dropna()
    ax.hist(data, bins=50, color=C['neutral'], alpha=0.8, edgecolor='#0F1117')
    # Linhas de média e mediana
    ax.axvline(data.mean(),   color=C['warn'],  lw=1.8, ls='--', label=f'Média={data.mean():.1f}')
    ax.axvline(data.median(), color=C['accent'],lw=1.8, ls=':',  label=f'Med={data.median():.1f}')
    ax.set_title(col, fontsize=11, color='#E6EDF3')
    ax.legend(fontsize=7, framealpha=0.2)
    ax.grid(alpha=0.2)
    # Skewness
    skew = data.skew()
    ax.text(0.97, 0.95, f'skew={skew:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color=C['muted'])

plt.suptitle('Distribuições Univariadas', fontsize=14,
             color=C['accent'], fontweight='bold')
plt.tight_layout(); plt.show()

---
### 1.5 Análise Bivariada (Feature × Default)

In [ ]:
# Distribuição de cada feature separada por classe de default
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.patch.set_facecolor('#0F1117')
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    for val, color, label in [(0, C['good'],'Bom (0)'), (1, C['bad'],'Default (1)')]:
        subset = df[df['default']==val][col].dropna()
        ax.hist(subset, bins=40, alpha=0.55, color=color,
                label=label, density=True, edgecolor='none')
    ax.set_title(col, fontsize=11, color='#E6EDF3')
    ax.legend(fontsize=8, framealpha=0.2)
    ax.grid(alpha=0.2)

plt.suptitle('Feature × Default (densidade normalizada)',
             fontsize=14, color=C['accent'], fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Boxplots: mediana e spread por classe
fig, axes = plt.subplots(3, 3, figsize=(15, 9))
fig.patch.set_facecolor('#0F1117')
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax    = axes[i]
    data0 = df[df['default']==0][col].dropna()
    data1 = df[df['default']==1][col].dropna()
    bp    = ax.boxplot([data0, data1], patch_artist=True, widths=0.5,
                       medianprops={'color':C['warn'],'linewidth':2.5})
    for patch, color in zip(bp['boxes'], [C['good'], C['bad']]):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticklabels(['Bom','Default'], fontsize=9)
    ax.set_title(col, fontsize=11, color='#E6EDF3')
    ax.grid(axis='y', alpha=0.2)
    # Diferença de medianas
    diff = abs(data1.median() - data0.median()) / data0.median() * 100
    ax.text(0.97, 0.97, f'Δmed={diff:.0f}%', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color=C['accent'])

plt.suptitle('Boxplots por Classe de Default',
             fontsize=14, color=C['accent'], fontweight='bold')
plt.tight_layout(); plt.show()

---
### 1.6 Correlações

In [ ]:
df_num = df[num_cols + ['default']].fillna(df.median(numeric_only=True))
corr   = df_num.corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0F1117')

# Heatmap completo
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap=sns.diverging_palette(220,20,as_cmap=True),
            vmax=0.8, vmin=-0.8, center=0, annot=True, fmt='.2f',
            annot_kws={'size':8}, square=True, linewidths=0.4,
            linecolor='#0F1117', ax=axes[0])
axes[0].set_title('Matriz de Correlação', fontsize=13, color='#E6EDF3')

# Correlação com o target
target_corr = corr['default'].drop('default').sort_values()
colors_bar  = [C['bad'] if v > 0 else C['good'] for v in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values,
             color=colors_bar, alpha=0.85, edgecolor='#0F1117')
axes[1].axvline(0, color='#8B949E', lw=1)
axes[1].set_title('Correlação com Default', fontsize=13, color='#E6EDF3')
axes[1].set_xlabel('Pearson r', fontsize=11)
axes[1].grid(axis='x', alpha=0.25)

plt.tight_layout(); plt.show()

print('\n  Top features correlacionadas com default:')
for feat, val in target_corr.abs().sort_values(ascending=False).items():
    bar = '█' * int(val*30)
    print(f'  {feat:<22} {bar:<30} {val:.3f}')

---
### 1.7 Insights e Decisões para o Pipeline

In [ ]:
default_rate = df['default'].mean()
top_corr     = corr['default'].drop('default').abs().idxmax()

print('╔══════════════════════════════════════════════════════════╗')
print('║  INSIGHTS DA EDA — DECISÕES PARA O PIPELINE            ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Dataset: {len(df):,} registros, {len(df.columns)} colunas           ║')
print(f'║  Target : {default_rate:.1%} de inadimplência                   ║')
print('║                                                          ║')
print('║  📌 Decisões técnicas derivadas da EDA:                 ║')
print('║                                                          ║')
print(f'║  1. DESBALANCEAMENTO ({default_rate:.0%} default)              ║')
print('║     → class_weight="balanced" na LR e RF                ║')
print('║     → scale_pos_weight no XGBoost                       ║')
print('║                                                          ║')
print('║  2. OUTLIERS (income, loan_amount com skew alto)        ║')
print('║     → Clipping por IQR na limpeza                       ║')
print('║     → log1p nas features muito skewed                   ║')
print('║                                                          ║')
print('║  3. NULOS (~3% em income, credit_score, employment)     ║')
print('║     → Imputação por mediana (robusta a outliers)        ║')
print('║     → Fit da mediana APENAS no treino                   ║')
print('║                                                          ║')
print(f'║  4. FEATURE MAIS CORRELACIONADA: {top_corr:<14}       ║')
print('║     → Criar features derivadas de domínio               ║')
print('║     → Ver Notebook 02_feature_engineering               ║')
print('║                                                          ║')
print('║  5. SEM COLUNAS CATEGÓRICAS                             ║')
print('║     → Encoding não necessário neste dataset             ║')
print('╚══════════════════════════════════════════════════════════╝')